In [2]:
from src.data import load_data
from src.preprocessing import preprocess_dataframe
from src.features import split_data, create_classical_features

df = load_data()
df_lemma,df_stemm = preprocess_dataframe(df)
X_train_lemma, X_test_lemma, y_train_lemma, y_test_lemma = \
    split_data(df_lemma, 'text_lemma')

X_train_stem, X_test_stem, y_train_stem, y_test_stem = \
    split_data(df_stemm, 'text_stem')

print("\nOriginal:")
print(df['text'].iloc[0])

print("\nClean:")
print(df['text_clean'].iloc[0])

print("\nLemma:")
print(df['text_lemma'].iloc[0])

print("\nStem:")
print(df['text_stem'].iloc[0])

['product_id', 'product_title', 'title_en', 'user_id', 'likes', 'dislikes', 'verification_status', 'recommend', 'title', 'comment', 'advantages', 'disadvantages']
(100000, 12)
['recommend', 'title', 'comment', 'advantages', 'disadvantages']
['\\N' 'recommended' 'not_recommended' 'no_idea']
1    واقعاً عالیه سلام، قبل اینکه نظرم رو بگم میخوا...
2    خیلی سخت حوله اش در میاد گیره های فلزی خیلی سخ...
3    گوشی مقرون به صرفه همه چیز در رابطه با ظاهر ای...
4    ابعاد، استحکام و نگهداری شارژ مناسب اگر ظرفیتش...
5    اقرار بیش از حد در ایراد گرفتن سلام دوستان،،_x...
Name: text, dtype: object
label
0    16110
1    10536
2    36972
Name: count, dtype: int64

Saved clean + lemma
Saved clean + stem
text_lemma: Train size: 50894, Test size: 12724
text_stem: Train size: 50894, Test size: 12724

Original:
واقعاً عالیه سلام، قبل اینکه نظرم رو بگم میخواستم به یک موضوع مهم اشاره کنم که نظراتی که ما برای کالاها ثبت میکنیم خیلی مهم هستن، چون بسیاری از مردم عزیز با استناد به این نظرات یک کالا رو خریداری م

In [3]:
from src.models.classical_models import run_classical_models
from src.evaluation import evaluate_model

# LEMMATIZATION

lemma_features = create_classical_features(X_train_lemma,X_test_lemma)
lemma_results = run_classical_models(
    lemma_features,
    y_train_lemma,
    y_test_lemma,
    evaluate_model
)

lemma_results['Preprocessing'] = 'Lemmatization'

display(
    lemma_results[
        [
            'Model',
            'Feature',
            'Accuracy',
            'Macro F1'
        ]
    ]
)


BOW Unigram: train (50894, 20000), test (12724, 20000)
BOW Bigram: train (50894, 20000), test (12724, 20000)
BOW Unigram+Bigram: train (50894, 20000), test (12724, 20000)
TFIDF Unigram: train (50894, 20000), test (12724, 20000)
TFIDF Bigram: train (50894, 20000), test (12724, 20000)
TFIDF Unigram+Bigram: train (50894, 20000), test (12724, 20000)


,Model,Feature,Accuracy,Macro F1
0,Naive Bayes,BOW Unigram+Bigram,0.779550,0.705268
1,Logistic Regression,TFIDF Unigram+Bigram,0.766111,0.704356
2,Linear SVM,TFIDF Unigram+Bigram,0.775778,0.688067
3,Logistic Regression,TFIDF Unigram,0.744263,0.683574
4,Naive Bayes,BOW Unigram,0.764932,0.677464
5,Logistic Regression,BOW Unigram+Bigram,0.750865,0.675921
6,Logistic Regression,TFIDF Bigram,0.735696,0.673083
7,Naive Bayes,BOW Bigram,0.769962,0.671948
8,Naive Bayes,TFIDF Unigram+Bigram,0.792518,0.669147
9,Linear SVM,TFIDF Unigram,0.760846,0.665174


In [4]:
# STEMMING

stem_features = create_classical_features(
    X_train_stem,
    X_test_stem
)

stem_results = run_classical_models(
    stem_features,
    y_train_stem,
    y_test_stem,
    evaluate_model
)

stem_results['Preprocessing'] = 'Stemming'

display(
    stem_results[
        [
            'Model',
            'Feature',
            'Accuracy',
            'Macro F1'
        ]
    ]
)


BOW Unigram: train (50894, 20000), test (12724, 20000)
BOW Bigram: train (50894, 20000), test (12724, 20000)
BOW Unigram+Bigram: train (50894, 20000), test (12724, 20000)
TFIDF Unigram: train (50894, 20000), test (12724, 20000)
TFIDF Bigram: train (50894, 20000), test (12724, 20000)
TFIDF Unigram+Bigram: train (50894, 20000), test (12724, 20000)


,Model,Feature,Accuracy,Macro F1
0,Logistic Regression,TFIDF Unigram+Bigram,0.770119,0.707297
1,Naive Bayes,BOW Unigram+Bigram,0.782930,0.707214
2,Linear SVM,TFIDF Unigram+Bigram,0.779943,0.691503
3,Naive Bayes,BOW Unigram,0.765876,0.679961
4,Logistic Regression,TFIDF Unigram,0.741905,0.679874
5,Naive Bayes,BOW Bigram,0.774599,0.676779
6,Logistic Regression,BOW Unigram+Bigram,0.748664,0.673435
7,Logistic Regression,TFIDF Bigram,0.737818,0.672495
8,Naive Bayes,TFIDF Unigram+Bigram,0.794561,0.671674
9,Linear SVM,TFIDF Unigram,0.764461,0.668763


In [5]:
import pandas as pd
top_results = (
    pd.concat([
        lemma_results,
        stem_results
    ])
    .sort_values(
        'Macro F1',
        ascending=False
    )
)

display(
    top_results[
        [
            'Preprocessing',
            'Model',
            'Feature',
            'Accuracy',
            'Macro F1'
        ]
    ].head(10)
)

,Preprocessing,Model,Feature,Accuracy,Macro F1
0,Stemming,Logistic Regression,TFIDF Unigram+Bigram,0.770119,0.707297
1,Stemming,Naive Bayes,BOW Unigram+Bigram,0.782930,0.707214
0,Lemmatization,Naive Bayes,BOW Unigram+Bigram,0.779550,0.705268
1,Lemmatization,Logistic Regression,TFIDF Unigram+Bigram,0.766111,0.704356
2,Stemming,Linear SVM,TFIDF Unigram+Bigram,0.779943,0.691503
2,Lemmatization,Linear SVM,TFIDF Unigram+Bigram,0.775778,0.688067
3,Lemmatization,Logistic Regression,TFIDF Unigram,0.744263,0.683574
3,Stemming,Naive Bayes,BOW Unigram,0.765876,0.679961
4,Stemming,Logistic Regression,TFIDF Unigram,0.741905,0.679874
4,Lemmatization,Naive Bayes,BOW Unigram,0.764932,0.677464
